In [1]:
import os 
import pandas as pd 
import numpy as np

### Loading the dataset sample_emails_with_triage_200.csv

In [2]:
df = pd.read_csv("../data/sample_emails_with_triage_200.csv") 
df.head()

,id,sender,subject,body,priority,triage_label
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond


### preparing the raw email text for further processing by creating a cleaned version of the email body.
Text cleaning is important because emails contain:punctuation,numbers,symbols,mixed casing,inconsistent whitespace

In [10]:
source_col = "body"

df['clean_text'] = ( 
df[source_col] 
.astype(str) 
.str.lower() 
.str.replace('[^a-zA-Z ]', '', regex=True) 
) 
df[[source_col, 'clean_text']].head()

,body,clean_text
0,Reminder: The client meeting is scheduled at 1...,reminder the client meeting is scheduled at t...
1,Your invoice of INR 25515.09 is due on 2025-12...,your invoice of inr is due on please pay to ...
2,Reminder: The client meeting is scheduled at 1...,reminder the client meeting is scheduled at t...
3,"Hello team, please find the attached weekly re...",hello team please find the attached weekly rep...
4,"Hello team, please find the attached weekly re...",hello team please find the attached weekly rep...


### Removing Stopwords and Extracting Useful Keywords
 After cleaning the email text, the next step is to extract meaningful keywords.


In [66]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words("english"))

df['keywords'] = df['clean_text'].apply(
    lambda x: [w for w in x.split() if w not in stop_words]
)

df[['clean_text', 'keywords']].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\theer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,clean_text,keywords
0,reminder the client meeting is scheduled at t...,"[reminder, client, meeting, scheduled, tomorro..."
1,your invoice of inr is due on please pay to ...,"[invoice, inr, due, please, pay, avoid, late, ..."
2,reminder the client meeting is scheduled at t...,"[reminder, client, meeting, scheduled, tomorro..."
3,hello team please find the attached weekly rep...,"[hello, team, please, find, attached, weekly, ..."
4,hello team please find the attached weekly rep...,"[hello, team, please, find, attached, weekly, ..."


### The purpose of the rule-based triage system is to classify each email into one of three categories:

notify_human – urgent or security-related messages

respond_or_act – emails requiring normal action

ignore – promotional or low-importance messages

In [67]:
def triage_rule(text):
    t = str(text)

    # simple rules (example)
    if any(k in t for k in ['invoice', 'payment due', 'due on', 'overdue']):
        return 'respond_or_act'

    if any(k in t for k in ['reset password', 'password reset', 'security issue']):
        return 'notify_human'

    if any(k in t for k in ['reset password','password reset','password']):
        return 'notify_human'
    # default fallback
    return 'respond_or_act'



In [68]:
df['clean_text']

0      reminder the client meeting is scheduled at  t...
1      your invoice of inr  is due on  please pay to ...
2      reminder the client meeting is scheduled at  t...
3      hello team please find the attached weekly rep...
4      hello team please find the attached weekly rep...
                             ...                        
195    security alert multiple failed login attempts ...
196    your order  has been shipped and is expected t...
197    congratulations you have been selected as a lu...
198    please complete the mandatory training module ...
199    congratulations you have been selected as a lu...
Name: clean_text, Length: 200, dtype: object

In [41]:
df['triage'] = df['clean_text'].apply(triage_rule)
df[['clean_text', 'triage']].head()

,clean_text,triage
0,reminder the client meeting is scheduled at t...,respond_or_act
1,your invoice of inr is due on please pay to ...,respond_or_act
2,reminder the client meeting is scheduled at t...,respond_or_act
3,hello team please find the attached weekly rep...,respond_or_act
4,hello team please find the attached weekly rep...,respond_or_act


In [42]:
df['triage'].value_counts()


triage
respond_or_act    184
notify_human       16
Name: count, dtype: int64

In [70]:
#debugging when your triage output looks wrong or unbalanced.
for kw in ['invoice','password','promotion']:
    print(kw, df['clean_text'].str.contains(kw).sum())


invoice 15
password 16
promotion 0


In [44]:
df['clean_text'].iloc[0:10].tolist()


['reminder the client meeting is scheduled at  tomorrow prepare your slides',
 'your invoice of inr  is due on  please pay to avoid late fees',
 'reminder the client meeting is scheduled at  tomorrow prepare your slides',
 'hello team please find the attached weekly report and action items for the project',
 'hello team please find the attached weekly report and action items for the project',
 'your invoice of inr  is due on  please pay to avoid late fees',
 'congratulations you have been selected as a lucky winner claim your prize now by clicking the link',
 'your order  has been shipped and is expected to deliver by ',
 'dear user we detected a login from a new device if this wasnt you reset your password immediately',
 'your invoice of inr  is due on  please pay to avoid late fees']

In [45]:
print(triage_rule(df['clean_text'].iloc[5]))


respond_or_act


# Improved triage function
This triage function takes the cleaned text of an email and checks for specific keywords in priority order.
It returns the appropriate label based on the first rule that matches.

### 1️.High-Priority Rules → notify_human

These rules detect urgent or sensitive messages such as:

invoice or overdue payments

password/security issues

fraud or suspicious login attempts
### 2.Medium-Priority Rules → respond_or_act

These rules identify normal work-related communication:

meetings and reminders

status updates

attachments and reports
### 3.Low-Priority Rules → ignore

These rules filter out marketing or spam-like emails:
### 4.Default Case

If none of the above rules match, the default label is:

return 'respond_or_act'

In [83]:
#improved function
def triage_rule(text):
    t = str(text)

    # 1. HIGH PRIORITY → notify_human
    if any(k in t for k in [
        'invoice', 'payment due', 'due on', 'overdue', 
        'password', 'reset password', 'password reset',
        'security issue', 'security alert', 
        'login attempt', 'failed login'
    ]):
        return 'notify_human'

    # 2. MEDIUM PRIORITY → respond_or_act
    if any(k in t for k in ['meeting', 'scheduled', 'reminder']):
        return 'respond_or_act'

    if 'report' in t:
        return 'respond_or_act'

    if any(k in t for k in ['hello team', 'attached', 'please find']):
        return 'respond_or_act'

    # 3. LOW PRIORITY → ignore
    if any(k in t for k in ['survey', 'promotion', 'sale', 'unsubscribe', 'offer']):
        return 'ignore'

    # 4. fallback
    return 'respond_or_act'


In [93]:
df['triage'] = df['clean_text'].apply(triage_rule)
df[['clean_text', 'triage']].head()

,clean_text,triage
0,reminder the client meeting is scheduled at t...,respond_or_act
1,your invoice of inr is due on please pay to ...,notify_human
2,reminder the client meeting is scheduled at t...,respond_or_act
3,hello team please find the attached weekly rep...,respond_or_act
4,hello team please find the attached weekly rep...,respond_or_act


## After improving the function we can categorize emails into three.

In [ ]:
df['triage'].value_counts()



triage
respond_or_act    128
notify_human       53
ignore             19
Name: count, dtype: int64

### Debug-friendly triage used from milestone1_template for more accuracy

In [96]:
# 3. Debug-friendly triage (regex-based with priority). This returns (label, matched_pattern)
RULES = {
    'notify_human': [
        r'\binvoice\b', r'\bbill\b', r'\bpayment\b', r'\bpassword\b', r'\breset\b',
        r'\bfraud\b', r'\bchargeback\b', r'\bdispute\b', r'\bdenied\b', r'\boverdue\b'
    ],
    'respond_or_act': [
        r'\bmeeting\b', r'\bschedule\b', r'\bscheduled\b', r'\battached\b', r'\battachment\b',
        r'\bplease find\b', r'\bplease review\b', r'\baction required\b', r'\burgent\b', r'\basap\b'
    ],
    'ignore': [
        r'\bunsubscribe\b', r'\bpromotion\b', r'\bsale\b', r'\boffer\b', r'\bnewsletter\b',
        r'\bsurvey\b', r'\bcongratulations\b'
    ]
}
COMPILED = {k:[re.compile(p, flags=re.I) for p in v] for k,v in RULES.items()}

def triage_rule_debug(text):
    if not isinstance(text, str) or text.strip()=="":
        return ('ignore','empty_text')
    for label in ['notify_human','respond_or_act','ignore']:
        for pat in COMPILED[label]:
            if pat.search(text):
                return (label, pat.pattern)
    return ('respond_or_act','default_fallback')

# apply and store both label and matched pattern
df[['triage','triage_match']] = df['clean_text'].apply(lambda t: pd.Series(triage_rule_debug(t)))
print("Triage value counts (advanced):")
print(df['triage'].value_counts())
display(df[['clean_text','triage','triage_match']].head(20))

Triage value counts (advanced):
triage
respond_or_act    127
ignore             42
notify_human       31
Name: count, dtype: int64


,clean_text,triage,triage_match
0,reminder the client meeting is scheduled at t...,respond_or_act,\bmeeting\b
1,your invoice of inr is due on please pay to ...,notify_human,\binvoice\b
2,reminder the client meeting is scheduled at t...,respond_or_act,\bmeeting\b
3,hello team please find the attached weekly rep...,respond_or_act,\battached\b
4,hello team please find the attached weekly rep...,respond_or_act,\battached\b
5,your invoice of inr is due on please pay to ...,notify_human,\binvoice\b
6,congratulations you have been selected as a lu...,ignore,\bcongratulations\b
7,your order has been shipped and is expected t...,respond_or_act,default_fallback
8,dear user we detected a login from a new devic...,notify_human,\bpassword\b
9,your invoice of inr is due on please pay to ...,notify_human,\binvoice\b


# Successfully saved the output in milestone1_Theertha.csv

In [97]:
out_name = "../data/milestone1_Theertha.csv"
df.to_csv(out_name, index=False)
print("Saved:", out_name)


Saved: ../data/milestone1_Theertha.csv
